In [ ]:
import math
import os
import sys
sys.path.append(os.path.abspath('.')) # to run files that are away
os.environ["WANDB_SILENT"] = "true"  # Suppress WandB logs

libraries = ["torch", "numpy", "polars"]
modules   = {lib: sys.modules.get(lib) for lib in libraries}

if not modules["torch"]:
    import torch
if not modules["numpy"]:
    import numpy as np
if not modules["polars"]:
    import polars as pl

from torch.utils.data import DataLoader, TensorDataset, random_split
import pandas as pd
import gc
import catboost as cb
import lightgbm as lgb
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import StandardScaler

from files_processor import LogFilesProcessor, WaferFilesProcessor, LogAndSpatialProcessor
from predictions import MultiOutputModelPredictor, PrePredictionProcessor
from asm_utils import count_missing_values_in_df, plot_all_columns_in_df, remove_constant_valued_cols, estimate_dataset_dimensionality, beep_sound

device = torch.device('mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu'))

import asm_data_wrangling as asm
from key_params import NUM_WAFERS, step_col_name, COMMON_ID_COLS, COMMON_ID_COLS_MOD, parquet_folder_name #, dict_of_spatial_files, dict_of_log_files 

log_processor = LogFilesProcessor(COMMON_ID_COLS_MOD, COMMON_ID_COLS)


In [ ]:
main_folder   = "../ASM_data"
dict_of_spatial_files = {'file1': {'path': f"{main_folder}/2. marathon0/Wafer performance/Spatial property after step 4.csv", 'marathon': 0},
                         'file2': {'path': f"{main_folder}/3. marathon1/Wafer performance/Spatial property.csv", 'marathon': 1}}

dict_of_log_files = {'file1': {'path': f"{main_folder}/2. marathon0/logs/Step1.csv", 'step': 1, 'marathon': 0},
                     'file2': {'path': f"{main_folder}/2. marathon0/logs/Step2.csv", 'step': 2, 'marathon': 0},
                     'file3': {'path': f"{main_folder}/2. marathon0/logs/Step3.csv", 'step': 3, 'marathon': 0},
                     'file4': {'path': f"{main_folder}/2. marathon0/logs/Step4.csv", 'step': 4, 'marathon': 0},
                     'file5': {'path': f"{main_folder}/3. marathon1/logs/Step1.csv", 'step': 1, 'marathon': 1},
                     'file6': {'path': f"{main_folder}/3. marathon1/logs/Step2.csv", 'step': 2, 'marathon': 1},
                     'file7': {'path': f"{main_folder}/3. marathon1/logs/Step3.csv", 'step': 3, 'marathon': 1},
                     'file8': {'path': f"{main_folder}/3. marathon1/logs/Step4.csv", 'step': 4, 'marathon': 1},}
marathon_run_col = "marathon_run"
wafer_col        = "wafer"
run_col          = "#run"
step_id_col      = "step_id"
process_time_col = "process time"

In [ ]:
"""code to split data into 4 wafers"""

# master_spatial_df, spatial_df_dict, y_df_dict, radius_wide_dict = asm.load_spatial_csv_and_create_targets(dict_of_spatial_files, main_folder, save=False)
# unique_marathon_runs_list = list(master_spatial_df["marathon_run"].unique())
# master_log_df = asm.load_and_process_and_combine_log_csv_files(dict_of_log_files, log_processor, unique_marathon_runs_list, step_col_name, main_folder, save=False)
# # master_log_df = master_log_df.fill_null(pl.lit(0))
# master_log_df = remove_constant_valued_cols(master_log_df)
# master_log_df = master_log_df.fill_null(pl.lit(0))

# # asm.dont_split_log_df_by_wafer_and_save_to_parquet(master_log_df, main_folder, overwrite = True)
# # asm.split_log_df_by_wafer_and_save_to_parquet(master_log_df, NUM_WAFERS, main_folder, log_processor, overwrite = True)

# # log_df_with_wafer_col = asm.infer_wafer_from_rc_values(master_log_df, rc_prefix="rc", wafer_col="wafer", overwrite = False)
# # log_df_with_wafer_col = asm.fast_infer_wafer(master_log_df, NUM_WAFERS)


In [ ]:
"""Code (new) to have a unified code/model (NOT split per wafer)"""

master_spatial_df, unique_marathon_runs_list = LogAndSpatialProcessor.load_spatial_csv_files_to_1_df(dict_of_spatial_files)
wide_radius_df, y_df = LogAndSpatialProcessor.create_target_df_from_spatial_df(master_spatial_df, main_folder, save=False)
del master_spatial_df
# ====
master_log_df = asm.load_and_process_and_combine_log_csv_files(dict_of_log_files, log_processor, unique_marathon_runs_list, step_col_name, main_folder, save=False)
master_log_df = remove_constant_valued_cols(master_log_df)
master_log_df = master_log_df.fill_null(pl.lit(0))
# ====
master_log_df_exploded = LogAndSpatialProcessor.explode_log_df_rows_by_wafer(master_log_df, NUM_WAFERS)
master_log_df_exploded = remove_constant_valued_cols(master_log_df_exploded)

del master_log_df


In [ ]:
# =========== reshape master_log_df_exploded here
# consider only 1 step
step_number    = 1
only_1_step_df = master_log_df_exploded.filter(pl.col(step_id_col) == step_number)

# subsample df
N_downsampling    = 200
subsampled_log_df = LogAndSpatialProcessor.downsample_df_rows(master_log_df_exploded, N_downsampling)

# latest rows per run:
num_last_rows_per_run = 100
latest_rows_per_run_df= (only_1_step_df.sort(process_time_col).group_by([marathon_run_col, wafer_col]).tail(num_last_rows_per_run))

reduced_log_df = subsampled_log_df
# ===========

log_df_with_no_constant_cols  = remove_constant_valued_cols(reduced_log_df)
joined_log_df_with_spatial_df = LogAndSpatialProcessor.join_radius_df_to_exploded_log_df(log_df_with_no_constant_cols, wide_radius_df)
y_df_expanded                 = LogAndSpatialProcessor.expand_y_df_to_match_size_of_log_df(joined_log_df_with_spatial_df, y_df)
joined_log_spatial_df_no_str  = joined_log_df_with_spatial_df.select(pl.exclude(pl.Utf8)) # remove str cols

#~~~~~~~~~~~~
preprocessor = PrePredictionProcessor()
X = joined_log_spatial_df_no_str.to_pandas().drop(columns=[wafer_col, marathon_run_col, run_col, run_col], errors='ignore')
y = y_df_expanded.drop(marathon_run_col, wafer_col).to_pandas()
# X_train, y_train, X_val, y_val, y_scaler = preprocessor.scale_and_split_data(X, y)
X_scaled, y_scaled, _ = preprocessor.scale_data_without_splitting(X, y)
X_train, X_val, y_train, y_val = train_test_split(X_scaled, y_scaled)
#~~~~~~~~~~~~


try:
    del only_1_step_df, log_df_with_no_constant_cols,\
        joined_log_df_with_spatial_df, master_log_df_exploded, subsampled_log_df#, wide_radius_df, y_df
except NameError:
    pass

In [ ]:
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from autoencoder import Autoencoder, TrainAutoencoder

this_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device from module: {this_device}")

layer1_dim=          128         # layer 1 nodes
layer2_dim=          64          # layer 2 nodes
latent_dim=          16          # num of features in latent layer, get this from data dimensionality
dropout_prob=        0.05
ae_training_epochs=  100        # training epochs
ae_batch_size=       126        # number of samples per batch
ae_optimizer_lr=     0.0011     # learning rate for the optimizer
weight_decay=        0.00001    # for L2 regularization
training_patience=   80         # how many epochs with no improvement to stop training
scheduler_patience=  80
scheduler_mode=      'min'      # min (max) reduces elarning rate when validation loss stops improving (starts increasing)
scheduler_factor=    0.8        # multiplies lr by this factor when validation loss plateaus

reduced_log_df_numeric = reduced_log_df.select(pl.col(pl.NUMERIC_DTYPES))

X_train_torch= torch.tensor(X_train.to_numpy(), dtype=torch.float32)
X_val_torch  = torch.tensor(X_val.to_numpy(),   dtype=torch.float32)

train_dataset = TensorDataset(X_train_torch, torch.zeros(len(X_train)))  # dummy labels
val_dataset   = TensorDataset(X_val_torch, torch.zeros(len(X_val)))

# Create DataLoader objects for train and validation datasets
input_size   = X_train_torch.shape[1]
train_loader = DataLoader(train_dataset, batch_size=ae_batch_size, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=ae_batch_size, shuffle=False)

autoencoder  = Autoencoder(input_size, layer1_dim, layer2_dim, latent_dim, dropout_prob)
optimizer    = torch.optim.AdamW(autoencoder.parameters(), lr=ae_optimizer_lr, weight_decay=weight_decay)
scheduler    = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, scheduler_mode, patience=scheduler_patience, factor=scheduler_factor)

trainer   = TrainAutoencoder()
best_loss = trainer.train_autoencoder(this_device, autoencoder, ae_training_epochs, train_loader, optimizer, scheduler,
                                      validation_loader=val_loader, patience = training_patience)
print(f"best loss: {best_loss:.3f}")


In [ ]:
from sklearn.decomposition import PCA

autoencoder.eval()

# Get latent space for X_train
with torch.no_grad():
    X_train_latent: np.ndarray = autoencoder.encoder(X_train_torch.to(this_device)).cpu().numpy()
    X_val_latent: np.ndarray   = autoencoder.encoder(X_val_torch.to(this_device)).cpu().numpy()
    
# 2D
plt.scatter(X_train_latent[:, 0], X_train_latent[:, 1], s=3, alpha=0.5)
plt.title("2D Latent Space")
plt.xlabel("Latent dim 1")
plt.ylabel("Latent dim 2")
plt.show()

# 3D
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_train_latent)

plt.scatter(X_pca[:, 0], X_pca[:, 1], s=3, alpha=0.5)
plt.title("Latent Space (PCA to 2D)")
plt.xlabel("PC 1")
plt.ylabel("PC 2")
plt.show()


In [ ]:
import seaborn as sns

# zero out z15
# X_train_latent[:, 8] = 0
X_train_recon = autoencoder.decoder(torch.from_numpy(X_train_latent).to(this_device)).detach().cpu().numpy()
X_val_recon   = autoencoder.decoder(torch.from_numpy(X_val_latent).to(this_device)).detach().cpu().numpy()

# ===

latent_df = pd.DataFrame(X_train_latent, columns=[f"z{i}" for i in range(X_train_latent.shape[1])])
target_df = pd.DataFrame(y_train, columns=[f"y{i}" for i in range(y_train.shape[1])])

# corr_matrix = latent_df.corr()
# sns.heatmap(corr_matrix.abs(), cmap='viridis')

corr_matrix = latent_df.corrwith(target_df, axis=0)  
# This won't work directly because corrwith compares series by index, not cross-columns.

# Instead, compute pairwise correlations manually:
corr_matrix = pd.DataFrame(
    np.corrcoef(latent_df.values.T, target_df.values.T)[:latent_dim, latent_dim:],
    index=latent_df.columns,
    columns=target_df.columns)

sns.heatmap(corr_matrix.abs(), cmap='viridis')
plt.xlabel('Targets')
plt.ylabel('Latent Features')
plt.show()


In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

kmeans = KMeans(n_clusters=5, random_state=42)
cluster_labels = kmeans.fit_predict(X_train_latent)

# Visualize
X_pca = PCA(n_components=2).fit_transform(X_train_latent)
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap='Set1', s=5)
plt.title("Clustering in Latent Space")
plt.show()

# # ===================
pca = PCA()
pca.fit(X_train)

threshold = 0.95
explained = pca.explained_variance_ratio_.cumsum()
plt.plot(explained)
plt.axhline(y=threshold, color='r', linestyle='--')

# =====================
n_components = np.argmax(explained >= threshold) + 1
print(f'Number of components: {n_components}')

top_components = pca.components_[:n_components]
print(top_components.shape)

# Top features for each component
top_features_idx = [np.argsort(np.abs(comp))[::-1] for comp in top_components]

# =====================
feature_names = X_train.columns  # if X_train is a pandas DataFrame

# top_k = 5
# # Compute percentage contribution per component
# for i, comp in enumerate(top_components):
#     abs_loadings = np.abs(comp)
#     percent_contrib = abs_loadings / abs_loadings.sum() * 100
#     top_indices = np.argsort(percent_contrib)[::-1]
#     print(f'\nComponent {i+1}:')
#     for idx in top_indices[:5]:  # Top 5 features
#         print(f'{feature_names[idx]}: {percent_contrib[idx]:.2f}%')

# =====================
from collections import defaultdict

feature_importance = defaultdict(float)

# Accumulate contribution of each feature across components
for comp in top_components:
    abs_loadings = np.abs(comp)
    percent_contrib = abs_loadings / abs_loadings.sum() * 100
    for i, contrib in enumerate(percent_contrib):
        feature_importance[feature_names[i]] += contrib

# Sort by total contribution
sorted_features = sorted(feature_importance.items(), key=lambda x: -x[1])

# Print top features
for feature, total_contrib in sorted_features[:30]:  # top 30
    print(f"{feature}: {total_contrib:.2f}%")



In [ ]:
def predict_catboost_single_model(X_train: np.ndarray, y_train: np.ndarray, X_val: np.ndarray, y_val: np.ndarray):
    """CatBoost only accepts uppercase 'task_type', beware of that"""
    y_train_np = y_train#.ravel()
    y_val_np   = y_val#.ravel()

    single_model = cb.CatBoostRegressor(iterations         = 50,
                                        learning_rate      = 0.4,
                                        depth              = 8,
                                        l2_leaf_reg        = 3,
                                        border_count       = 128,
                                        bagging_temperature= 0,
                                        task_type          = 'CPU',
                                        verbose            = 0,
                                        random_seed        = 42)
    single_model.fit(X_train, y_train_np)
    # Feature importance
    importances = single_model.get_feature_importance()

    # Prediction + RMSE
    y_pred_cat = single_model.predict(X_val)
    rmse_cat   = mean_squared_error(y_val_np, y_pred_cat) ** 0.5
    return rmse_cat, y_pred_cat, importances


def predict_catboost_multi(X_train: np.ndarray, y_train: np.ndarray, X_val: np.ndarray, y_val: np.ndarray):
    """CatBoost only accepts uppercase 'task_type', beware of that"""
    single_model = cb.CatBoostRegressor(iterations         = 50,
                                        learning_rate      = 0.4,
                                        depth              = 8,
                                        l2_leaf_reg        = 3,
                                        border_count       = 128,
                                        bagging_temperature= 0,
                                        task_type          = 'CPU',
                                        verbose            = 0,
                                        random_seed        = 42)
    multi_model  = MultiOutputRegressor(single_model)
    multi_model.fit(X_train, y_train)

    importances  = np.array([est.get_feature_importance() for est in multi_model.estimators_])
    y_pred_cat   = multi_model.predict(X_val)
    rmse_cat     = mean_squared_error(y_val, y_pred_cat) ** 0.5
    return rmse_cat, y_pred_cat, importances


def _make_predictions_in_1_function(X_train, y_train, X_val, y_val, device):
    predictor = MultiOutputModelPredictor(device)

    print(f"======== Results =======")
    # rmse_linreg, _ = predictor.predict_linear_reg(X_train, y_train, X_val, y_val)
    # print(f"LinReg RMSE: {rmse_linreg:.3f}")

    # rmse_ridge, _ = predictor.predict_linear_reg_ridge(X_train, y_train, X_val, y_val)
    # print(f"Ridge RMSE: {rmse_ridge:.3f}")

    # rmse_lgb, _ = predictor.predict_lightgbm(X_train, y_train, X_val, y_val)
    # print(f"LGBM RMSE: {rmse_lgb:.3f}")

    rmse_cat, _ , importances = predictor.predict_catboost(X_train, y_train, X_val, y_val)
    print(f"Catboost RMSE: {rmse_cat:.3f}")

    # rmse_rf, _  = predictor.predict_randomforest(X_train, y_train, X_val, y_val)
    # print(f"RF RMSE: {rmse_rf:.3f}")


In [ ]:
# Get error on RECONSTRUCTED X (using autoencoder)

# rmse_cat, y_pred_cat, importances = predict_catboost_multi(X_train_latent, y_train, X_val_latent, y_val)
# print(rmse_cat)#, y_pred_cat)

rmse_cat, y_pred_cat, importances = predict_catboost_multi(X_train_recon, y_train, X_val_recon, y_val)
print(rmse_cat)#, y_pred_cat)

# Conclusion: compressing to latent, zeroing the least correlated latent col, then decompressing and predicting yields WORSE score

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import make_scorer, mean_squared_error

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))
rmse_scorer = make_scorer(rmse, greater_is_better=False)

def train_models_in_1_dataset(y_df_expanded, joined_log_spatial_df, main_folder, device):
    """Edited to do everything in 1 dataset"""
    marathon_run_col = "marathon_run"
    wafer_col        = "wafer"
    run_col          = "#run"

    # convert wafer col to str, maybe it helps:
    # joined_log_spatial_df = joined_log_spatial_df.with_columns(pl.col(wafer_col).cast(str))

    preprocessor = PrePredictionProcessor()

    X_train: pd.DataFrame
    y_train: np.ndarray
    X_val:   pd.DataFrame
    y_val:   np.ndarray

    want_to_scale_per_wafer = False
    if want_to_scale_per_wafer:
        X = joined_log_spatial_df.to_pandas().drop(columns=[marathon_run_col, run_col], errors='ignore')
        y = y_df_expanded.drop(marathon_run_col).to_pandas()
        X_train, y_train, X_val, y_val, wafer_x_scalers, wafer_y_scalers = preprocessor.scale_per_wafer_and_split_data(X, y, wafer_col="wafer", test_size=0.2)
    else:
        X = joined_log_spatial_df.to_pandas().drop(columns=[wafer_col, marathon_run_col, run_col], errors='ignore')
        y = y_df_expanded.drop(marathon_run_col, wafer_col).to_pandas()
        # X_train, y_train, X_val, y_val, y_scaler = preprocessor.scale_and_split_data(X, y)
        X_scaled, y_scaled, _ = preprocessor.scale_data_without_splitting(X, y)
        X_train, X_val, y_train, y_val = train_test_split(X_scaled, y_scaled)


    numeric_cols  = X_train.select_dtypes(include=[np.number]).columns
    zero_var_cols = X_train[numeric_cols].columns[X_train[numeric_cols].var() == 0].tolist()
    X_train_clean = X_train.drop(columns = zero_var_cols)
    X_val_clean   = X_val.drop(columns = zero_var_cols)

    # =-=-=-=-=-= bit about feature importance
    importances = _make_predictions_in_1_function(X_train_clean, y_train, X_val_clean, y_val, device)
    top_features_fraction = 0.05
    X_train_clean_light, X_val_clean_light = LogAndSpatialProcessor.keep_top_features_by_importance(X_train_clean, X_val_clean, importances, top_features_fraction)
    
    # Cross Val score
    model  = RandomForestRegressor()
    scores = cross_val_score(model, X_train_clean_light, y_train, scoring=rmse_scorer, cv=5)
    print(f"CV RMSE mean: {-scores.mean():.3f}")
    
    importances = _make_predictions_in_1_function(X_train_clean_light, y_train, X_val_clean_light, y_val, device)
    # =-=-=-=-=-=-=-=-=-=-=-=

    # ========= bit about correlation
    # correlations = pd.DataFrame({
    #     f"target_{i}": X_train_clean.corrwith(pd.Series(y_train[:, i], index=X_train_clean.index)).abs()
    #     for i in range(y_train.shape[1])})

    # # Average correlations across all targets
    # mean_correlations = correlations.mean(axis=1)

    # # Model importances (make sure index aligns with X_train_clean.columns)
    # importances_mean = pd.Series(mean_importance, index=X_train_clean.columns)

    # # Plot correlation vs importance
    # plt.figure(figsize=(8,6))
    # plt.scatter(mean_correlations, importances_mean)
    # plt.xlabel("Mean Abs(Correlation) with Targets")
    # plt.ylabel("Mean Model Feature Importance")
    # plt.title("Feature Importance vs. Correlation")
    # plt.grid(True)
    # plt.show()
    # ============

    return importances

importances = train_models_in_1_dataset(y_df_expanded, joined_log_spatial_df_no_str, main_folder, device)


In [ ]:
# ['rc3 signal_1_step4', 'common signal_88_step4', 'common signal_64_step4', 'common signal_92_step4', 'common signal_91_step4', 'common signal_90_step4', 'common signal_87_step4', 'common signal_52_step4', 'common signal_85_step4', 'common signal_86_step4', 'common signal_43_step4', 'common signal_61_step4', 'rc3 signal_0_step4', 'rc3 signal_3_step4', 'common signal_58_step4', 'rc3 signal_2_step4', 'common signal_89_step4', 'common signal_60_step4', 'common signal_70_step4', 'common signal_62_step4', 'common signal_54_step4', 'common signal_69_step4', 'common signal_44_step4', 'common signal_63_step4', 'rc4 signal_0_step4', 'common signal_83_step4', 'common signal_55_step4', 'rc1 signal_0_step4', 'common signal_57_step4', 'common signal_111_step4', 'common signal_48_step4', 'common signal_82_step4', 'rc1 signal_1_step4', 'common signal_80_step4', 'common signal_84_step4', 'common signal_65_step4', 'rc4 signal_2_step4', 'rc2 signal_5_step4', 'common signal_40_step4', 'rc2 signal_6_step4', 'common signal_157_step4', 'rc2 signal_2_step4', 'rc2 signal_1_step4', 'common signal_78_step4', 'rc1 signal_3_step4', 'rc1 signal_6_step4', 'rc1 signal_5_step4', 'rc4 signal_3_step4', 'common signal_39_step4', 'common signal_73_step4', 'rc4 signal_5_step4', 'rc1 signal_2_step4', 'rc2 signal_3_step4', 'rc4 signal_6_step4', 'common signal_71_step4', 'common signal_103_step4', 'common signal_122_step4', 'rc2 signal_0_step4', 'rc3 signal_6_step4', 'common signal_93_step4', 'common signal_94_step4', 'rc4 signal_1_step4', 'common signal_41_step4', 'common signal_121_step4', 'common signal_49_step4', 'common signal_123_step4', 'common signal_113_step4', 'common signal_50_step4', 'common signal_155_step4', 'common signal_104_step4', 'common signal_108_step4', 'common signal_51_step4', 'common signal_153_step4', 'common signal_98_step4', 'common signal_45_step4', 'common signal_95_step4', 'common signal_59_step4', 'common signal_74_step4', 'rc3 signal_5_step4', 'common signal_14_step4', 'common signal_97_step4', 'common signal_77_step4', 'common signal_118_step4', 'common signal_156_step4', 'common signal_96_step4', 'common signal_107_step4', 'common signal_99_step4', 'common signal_154_step4', 'common signal_115_step4', 'common signal_53_step4', 'common signal_120_step4', 'common signal_102_step4', 'common signal_110_step4', 'common signal_105_step4', 'common signal_46_step4', 'common signal_119_step4', '104', 'common signal_114_step4', 'common signal_112_step4', 'common signal_109_step4', 'common signal_100_step4', 'common signal_79_step4', 'common signal_36_step4']

plt.figure(figsize=(17, 8))
# y_values = joined_log_spatial_df_no_str['rc3 signal_1_step4']
y_values = joined_log_spatial_df_no_str['common signal_88_step4']
x_values = range(len(y_values))
plt.scatter(x_values, y_values)
plt.show()


In [ ]:
which_row = 1

fig, ax = plt.subplots()
ax.scatter(range(len(y_val[0])), y_val[which_row], label='real')
ax.scatter(range(len(y_val[0])), y_pred_cat[which_row], label='predicted')
ax.set_xlabel("Site ID")
ax.set_ylabel("Spatial property (nm)")
ax.set_title("Spatial property, real vs predicted")
ax.legend()
plt.show()

In [ ]:
which_row = 1
y_real_values = (y_df_expanded.row(which_row))[2:]

y_pred_cat_original_scale = y_scaler.inverse_transform(y_pred_cat)
y_val_original_scale = y_scaler.inverse_transform(y_val)
y_pred_values = y_val_original_scale[which_row]

fig, ax = plt.subplots(figsize=(10, 5.5))
ax.scatter(range(len(y_real_values)), y_real_values, label='real')
ax.scatter(range(len(y_pred_values)), y_pred_values, label='predicted')
ax.set_xlabel("Site ID")
ax.set_ylabel("Spatial property (nm)")
ax.set_title("Spatial property, real vs predicted")
ax.legend()
plt.show()


In [ ]:
# code just to break the run
beep_sound(4)

x-1

# col "common signal_49_step4" has big outliers

In [ ]:
"""Hyperparam search"""

predictor    = MultiOutputModelPredictor(device)
preprocessor = PrePredictionProcessor()

marathon_run_col = "marathon_run"
wafer_col        = "wafer"

y = y_df_expanded.drop(marathon_run_col).to_pandas()
X = joined_log_df_with_spatial_df.to_pandas().drop(columns=[wafer_col, marathon_run_col], errors='ignore')
X = preprocessor.drop_certain_cols_from_df(X, [marathon_run_col])

X_train: pd.DataFrame
y_train: np.ndarray
X_val:   pd.DataFrame
y_val:   np.ndarray
# X_train, y_train, X_val, y_val, y_scaler = preprocessor.scale_and_split_data(X, y)
X_scaled, y_scaled, _ = preprocessor.scale_data_without_splitting(X, y)
X_train, X_val, y_train, y_val = train_test_split(X_scaled, y_scaled)

numeric_cols  = X_train.select_dtypes(include=[np.number]).columns
zero_var_cols = X_train[numeric_cols].columns[X_train[numeric_cols].var() == 0].tolist()
X_train = X_train.drop(columns=zero_var_cols)
X_val   = X_val.drop(columns=zero_var_cols)

# best_model = predictor.tune_catboost_hyperparams(X_train, y_train)
best_model = predictor.tune_lightgbm_hyperparams(X_train, y_train)


In [ ]:
for i in range(5):
    beep_sound()

# 0.4 --> 0.1023
# 0.5 --> 0.0978

# Combo 1/8: est=50, lr=0.46, depth=8, lambda=3, leaves=64, bag_frac=1.0
# Avg RMSE: 0.0833

# Combo 5/12: est=50, lr=0.4, depth=6, lambda=3, leaves=64, bag_frac=1.0
# Avg RMSE: 0.0822


In [ ]:
def join_features_targets(big_log_df, target_df):
    # target columns except marathon_run
    target_cols = [str(i) for i in range(1, 110)]
    
    # merge on marathon_run
    full_df = big_log_df.merge(target_df, on='marathon_run', how='left')
    
    # features: drop target columns + marathon_run if not needed as feature
    X = full_df.drop(columns=target_cols + ['marathon_run'])
    
    # targets
    y = full_df[target_cols]
    
    return X, y


def train_one_model_for_all_wafers(y_df_dict, radius_wide_dict, big_log_df, marathon_run_col, target_cols, device):
    predictor    = MultiOutputModelPredictor(device)
    preprocessor = PrePredictionProcessor()
    wafer_col    = "wafer"

    # Combine all y dfs into one with wafer column
    y_dfs = []
    for k, df in y_df_dict.items():
        y_dfs.append(df.with_columns(pl.lit(k+1).alias(wafer_col)))
    combined_y_df = pl.concat(y_dfs, how="vertical")

    # Combine all radius dfs into one with wafer column
    radius_dfs = []
    for k, df in radius_wide_dict.items():
        radius_dfs.append(df.with_columns(pl.lit(k+1).alias(wafer_col)))
    combined_radius_df = pl.concat(radius_dfs, how="vertical")

    # Flatten last rows of big_log_df by marathon_run
    processed_log_df = asm._flatten_last_n_rows(big_log_df, marathon_run_col, num_of_last_rows=1)

    # Join features with radius info on marathon_run and wafer
    features_df = processed_log_df.join(
        combined_radius_df,
        on  = [marathon_run_col, wafer_col],
        how = "left")

    # Join features with targets on marathon_run and wafer
    full_df = features_df.join(
        combined_y_df,
        on  = [marathon_run_col, wafer_col],
        how = "inner")

    # Prepare X and y
    y = full_df.select(target_cols).to_pandas()
    X = full_df.drop(target_cols + [marathon_run_col, wafer_col]).to_pandas()

    X = preprocessor.drop_certain_cols_from_df(X, [marathon_run_col])

    # Scale and split
    # X_train, y_train, X_val, y_val, y_scaler = preprocessor.scale_and_split_data(X, y)
    X_scaled, y_scaled, _ = preprocessor.scale_data_without_splitting(X, y)
    X_train, X_val, y_train, y_val = train_test_split(X_scaled, y_scaled)
    X_train = X_train.fillna(0)
    X_val   = X_val.fillna(0)

    # Drop zero variance cols
    zero_var_cols = X_train.columns[X_train.var() == 0].tolist()
    X_train = X_train.drop(columns=zero_var_cols)
    X_val   = X_val.drop(columns=zero_var_cols)

    # Train all models and print results
    rmse_linreg, _ = predictor.predict_linear_reg(X_train, y_train, X_val, y_val)
    rmse_ridge, _  = predictor.predict_linear_reg_ridge(X_train, y_train, X_val, y_val)
    rmse_lgb, _    = predictor.predict_lightgbm(X_train, y_train, X_val, y_val)
    # rmse_cat, _ = predictor.predict_catboost(X_train, y_train, X_val, y_val)
    rmse_rf, _     = predictor.predict_randomforest(X_train, y_train, X_val, y_val)

    print(f"LinReg RMSE: {rmse_linreg:.3f}")
    print(f"Ridge RMSE: {rmse_ridge:.3f}")
    print(f"LGBM RMSE: {rmse_lgb:.3f}")
    # print(f"Catboost RMSE: {rmse_cat:.3f}")
    print(f"RF RMSE: {rmse_rf:.3f}")

    return {"linreg_rmse": rmse_linreg,
            "ridge_rmse":  rmse_ridge,
            "lgbm_rmse":   rmse_lgb,
            # "catboost_rmse": rmse_cat,
            "rf_rmse":     rmse_rf,}

target_cols = [str(i) for i in range(1, 110)]  # '1' to '109'

results = train_one_model_for_all_wafers(
    y_df_dict        = y_df_dict,
    radius_wide_dict = radius_wide_dict,
    big_log_df       = big_log_df,
    marathon_run_col = "marathon_run",
    target_cols      = target_cols,
    device           = device)

print(results)

# combined_log_df["wafer"]
# print(y_df_dict[0].columns)


In [ ]:

# df = pl.read_csv(f'../ASM_data/2. marathon0/logs/Step1.csv', null_values=["", "NA", "null"], ignore_errors=True, separator=",")
pdf         = pd.read_csv(f'../ASM_data/2. marathon0/logs/Step4.csv', decimal='.')
pdf_numeric = pdf.select_dtypes(include='number') # removes strings
df          = pl.from_pandas(pdf_numeric)

df.head()

In [ ]:
# plot_all_columns_in_df(pdf[pdf.columns[:10]])

df2 = remove_constant_valued_cols(df)
df_scaled = StandardScaler().fit_transform(df2)

num_rows_subsample = 500_000
if df_scaled.shape[0] > num_rows_subsample:
    idx = np.random.choice(df_scaled.shape[0], num_rows_subsample, replace=False)
    df_scaled_sample = df_scaled[idx]
else:
    df_scaled_sample = df_scaled

df_scaled = pl.DataFrame(df_scaled)  # Only do this if df_scaled is a NumPy array
# plot_all_columns_in_df(df_scaled.select(df_scaled.columns[:20]))
# plot_all_columns_in_df(df2.select(df2.columns[:20]))


# # plt.plot(df2[df2.columns[3]].to_numpy()[:200_000])
# y1 = df2[df2.columns[3]].to_numpy()[:5_0000]
# x1 = np.arange(len(y1))
# plt.scatter(x1, y1)#, s=1)  # s=1 for smaller points


def find_square_periodicity_of_feature(signal):
    """Requires a square function"""
    durations   = np.diff(np.where(np.diff(signal) != 0)[0])
    periods     = durations[::2] + durations[1::2]  # Sum long+short phases
    estimated_p = int(np.round(np.median(periods)))
    return estimated_p

def sin_encode(signal, periodicity):
    sin_encoded = np.sin(2 * np.pi * signal/periodicity)
    return sin_encoded

signal = df2[:, 3].to_numpy()
periodicity = find_square_periodicity_of_feature(signal)
sin_encoded = sin_encode(signal, periodicity)
print(periodicity)
print(sin_encoded)

plt.plot(sin_encoded[:20_000])

# p = 1
# sin_encoded = np.sin(2* np.pi* x/p)

# data_dimensionality = estimate_dataset_dimensionality(df_scaled_sample, 150)
# print(f"Recommended latent layer size: {data_dimensionality:.1f}")

# beep_sound(3, 0)

In [ ]:
import umap
reducer   = umap.UMAP()
embedding = reducer.fit_transform(df_scaled)
embedding.shape



In [ ]:

main_folder   = "../ASM_data"
dict_of_spatial_files = {'file1': {'path': f"{main_folder}/2. marathon0/Wafer performance/Spatial property after step 4.csv", 'marathon': 0},
                       'file2': {'path': f"{main_folder}/3. marathon1/Wafer performance/Spatial property.csv", 'marathon': 1}}

dict_of_log_files = {'file1': {'path': f"{main_folder}/2. marathon0/logs/Step1.csv", 'step': 1, 'marathon': 0},
                     'file2': {'path': f"{main_folder}/2. marathon0/logs/Step2.csv", 'step': 2, 'marathon': 0},
                     'file3': {'path': f"{main_folder}/2. marathon0/logs/Step3.csv", 'step': 3, 'marathon': 0},
                     'file4': {'path': f"{main_folder}/2. marathon0/logs/Step4.csv", 'step': 4, 'marathon': 0},
                     'file5': {'path': f"{main_folder}/3. marathon1/logs/Step1.csv", 'step': 1, 'marathon': 1},
                     'file6': {'path': f"{main_folder}/3. marathon1/logs/Step2.csv", 'step': 2, 'marathon': 1},
                     'file7': {'path': f"{main_folder}/3. marathon1/logs/Step3.csv", 'step': 3, 'marathon': 1},
                     'file8': {'path': f"{main_folder}/3. marathon1/logs/Step4.csv", 'step': 4, 'marathon': 1},}

# def train_models(y_df_dict, radius_wide_dict, main_folder, num_wafers, device):
#     predictor    = MultiOutputModelPredictor(device)
#     preprocessor = PrePredictionProcessor()

#     marathon_run_col = "marathon_run"
#     wafer_col        = "wafer"

#     total_rmse_lgb, total_rmse_cat, total_rmse_rf, total_rmse_linreg, total_rmse_ridge = 0, 0, 0, 0, 0
#     for wafer_idx in range(num_wafers):
#         wafer_log_df             = pl.read_parquet(f"{main_folder}/{parquet_folder_name}/{wafer_col}_{wafer_idx+1}_log.parquet")
#         # processed_wafer_log_df   = _compute_log_df_grouped_stats(wafer_log_df, 'marathon_run')
#         processed_wafer_log_df   = asm._flatten_last_n_rows(wafer_log_df, marathon_run_col)
#         break
#     return processed_wafer_log_df

master_spatial_df, spatial_df_dict, y_df_dict, radius_wide_dict = asm.load_spatial_csv_and_create_targets(dict_of_spatial_files, main_folder, save=False)
unique_marathon_runs_list = list(master_spatial_df["marathon_run"].unique())
master_log_df = asm.load_and_process_and_combine_log_csv_files(dict_of_log_files, log_processor, unique_marathon_runs_list, step_col_name, main_folder, save=False)
master_log_df = remove_constant_valued_cols(master_log_df)
# asm.split_log_df_by_wafer_and_save_to_parquet(master_log_df, NUM_WAFERS, main_folder, log_processor, overwrite = False)
# processed_wafer_log_df = train_models(y_df_dict, radius_wide_dict, main_folder, NUM_WAFERS, device)



In [ ]:
df1 = pd.read_csv(f"{main_folder}/2. marathon0/logs/Step4.csv")
df2 = pd.read_csv(f"{main_folder}/3. marathon1/logs/Step4.csv")

print(df1.shape)
print(df2.shape)


In [ ]:
def read_csv_and_lowercase_cols_names(file_path: str) -> pl.DataFrame:
    """Read CSV into Polars DataFrame and title-case column names after stripping spaces"""
    df = pl.read_csv(file_path, ignore_errors=True)
    df = df.rename({c: c.strip().title() for c in df.columns})
    return df

count_missing_values_in_df(dict_of_log_files[1])


In [ ]:
y_pred_cat.shape
y_pred_unscaled = y_scaler.inverse_transform(y_pred_cat)

type(y_pred_cat)

In [ ]:
y_pred_unscaled = y_scaler.inverse_transform(y_pred_cat)

plt.scatter(range(len(y_full_pd.iloc[1].values)), y_full_pd.iloc[1].values,
            label="True", facecolors='none', edgecolors='blue', s=8)
plt.scatter(range(len(y_pred_unscaled[1])), y_pred_unscaled[1],
            label="Predicted", facecolors='none', edgecolors='orange', s=8)
# plt.plot(y_full_pd.iloc[1].values, label="True")
# plt.plot(y_pred_unscaled[1], label="Predicted")
plt.legend()
plt.title("y_predicted vs y_actual")
plt.xlabel("Site ID (coordinate)")
plt.ylabel("Spatial property")
plt.show()


In [ ]:
# hyperparam search

from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'estimator__num_leaves': [20, 31, 40, 50],
    'estimator__max_depth': [-1, 5, 10, 20],
    'estimator__min_data_in_leaf': [10, 20, 30],
    'estimator__learning_rate': [0.01, 0.05, 0.1],
    'estimator__n_estimators': [100, 500, 1000]}

# model = MultiOutputRegressor(xgb.XGBRegressor(objective='reg:squarederror', verbosity=0))
model = MultiOutputRegressor(LGBMRegressor(objective='regression', verbosity=-1))

search = RandomizedSearchCV(
    model,
    param_distributions=param_dist,
    n_iter=20,
    scoring='neg_mean_squared_error',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1)

search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV RMSE:", (-search.best_score_)**0.5)



##### Spatial data (M)

In [ ]:
def plot_wafer_property(df, property_col, title):
    plt.figure(figsize=(9, 5))
    for rc_value, group in df.group_by("RC"):
        x = group["#Run"].to_list()
        y = group[property_col].to_list()
        plt.scatter(x, y, label=f'RC {rc_value}', s=20)
    plt.xlabel("#Run")
    plt.ylabel(property_col)
    plt.title(title)
    plt.legend()
    plt.show()

plot_wafer_property(wafer_df, "Wafer property summary 1", "Wafer Property 1")
plot_wafer_property(wafer_df, "Wafer property summary 2", "Wafer Property 2")


##### Import timeseries data (S)

In [ ]:
"""Load data and make parquet files out of it"""

should_we_save_parquet_files = False

def _save_df_as_parquet_file(df: pl.dataframe, saving_location: str):
    df.write_parquet(saving_location)

def remove_unchanging_cols_from_df_and_save(df: pl.dataframe, col_name: str, should_we_save_parquet_files: bool) -> None:
    for run_id in df[col_name].unique().to_list():
        df_per_run    = df.filter(pl.col(col_name) == run_id)
        constant_cols = [col for col in df_per_run.columns
                     if df_per_run[col].dtype in [pl.Int64, pl.Float64] and
                        #  df_per_run.select(pl.col(col).filter(pl.col(col) != 0)).height == 0]
                         df_per_run.select(pl.col(col).n_unique()).item() == 1]
        df_per_run_filtered = df_per_run.drop(constant_cols)
        saving_location = f"{parquet_subfolder}/run_{run_id}.parquet"
        if should_we_save_parquet_files:
            _save_df_as_parquet_file(df_per_run_filtered, saving_location)

remove_unchanging_cols_from_df_and_save(log_df, "#Run", should_we_save_parquet_files)

# =============
# before making funcrtion:

# if should_we_save_parquet_files:
#     for run_id in log_df["#Run"].unique().to_list():
#         df_per_run = log_df.filter(pl.col("#Run") == run_id)
#         zero_cols  = [col for col in df_per_run.columns
#                      if df_per_run[col].dtype in [pl.Int64, pl.Float64] and
#                         #  df_per_run.select(pl.col(col).filter(pl.col(col) != 0)).height == 0]
#                          df_per_run.select(pl.col(col).n_unique()).item() == 1]
#         df_per_run_filtered = df_per_run.drop(zero_cols)
#         df_per_run_filtered.write_parquet(f"{parquet_subfolder}/run_{run_id}.parquet")


##### Timeseries data (S)

In [ ]:
run_number   = 15
parquet_file = f"./ASM_data/3. marathon1/Logs/split_by_run/run_{run_number}.parquet"
df           = pl.read_parquet(parquet_file)
df_pd        = df.to_pandas()
df_numeric   = df.select(pl.col(pl.NUMERIC_DTYPES))
X_np         = df_numeric.to_numpy()
X_scaled     = StandardScaler().fit_transform(X_np)

num_cols_to_plot = len(df_pd.columns)
num_rows         = math.ceil(math.sqrt(num_cols_to_plot))
num_cols_grid    = math.ceil(num_cols_to_plot / num_rows)

axes = df_pd.plot(subplots=True, figsize=(14, 12), layout=(num_rows, num_cols_grid), sharex=True, legend=False)

if isinstance(axes, np.ndarray):
    axes_flat = axes.flatten()
else:
    axes_flat = [axes]

column_names = df_pd.columns.tolist()

for i, ax in enumerate(axes_flat):
    if i < num_cols_to_plot: # Only set title for actual plots
        ax.set_title(column_names[i], fontsize='xx-small')
    ax.set_xlabel('')
    ax.set_ylabel('')
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_xticklabels([])
    ax.set_yticklabels([])

for i in range(num_cols_to_plot, len(axes_flat)):
    axes_flat[i].set_visible(False)

plt.tight_layout()
plt.suptitle(f'Run #{run_number} {log_file}', fontsize='large', y=1.02) # Adjust y to prevent overlap
plt.show()